# Azure ML - Databricks MLflow Model Integration Test

This notebook validates that **Azure ML can load and run a model from Databricks MLflow registry**.

## Prerequisites
- Databricks workspace with a registered MLflow model
- Azure ML managed identity with permissions to access Databricks
- Required packages: `databricks-sdk`, `mlflow`, `numpy`, `pandas`

## Required Environment Variables
- `DATABRICKS_HOST`: Databricks workspace URL (e.g., `https://adb-<id>.<region>.azuredatabricks.net`)
- `DATABRICKS_MODEL_NAME`: MLflow model name (e.g., `sklearn-model`)
- `DATABRICKS_MODEL_VERSION`: Model version or "latest" (optional, defaults to latest)

## Test Flow
1. Authenticate to Databricks using Azure managed identity
2. Load MLflow model from Databricks registry
3. Make predictions with sample data
4. Validate predictions

In [ ]:
import os
import json
import logging
from typing import Any, Dict, List

import numpy as np
import pandas as pd

# Azure and Databricks SDKs
from azure.identity import DefaultAzureCredential, ClientSecretCredential, ChainedTokenCredential, EnvironmentCredential
from databricks.sdk import WorkspaceClient
import mlflow
import mlflow.pyfunc

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✓ All required packages imported successfully")

## 1. Authenticate to Databricks

In [ ]:
# Get Databricks connection details from environment
databricks_host = os.getenv("DATABRICKS_HOST")
databricks_model_name = os.getenv("DATABRICKS_MODEL_NAME")
databricks_model_version = os.getenv("DATABRICKS_MODEL_VERSION", "latest")

if not all([databricks_host, databricks_model_name]):
    raise ValueError(
        "Missing required environment variables:\n"
        "  - DATABRICKS_HOST: Databricks workspace URL\n"
        "  - DATABRICKS_MODEL_NAME: MLflow model name to load"
    )

logger.info(f"Databricks Host: {databricks_host}")
logger.info(f"Model Name: {databricks_model_name}")
logger.info(f"Model Version: {databricks_model_version}")

In [ ]:
# Authenticate to Databricks using Azure managed identity or service principal
tenant_id = os.getenv("AZURE_TENANT_ID")
client_id = os.getenv("AZURE_CLIENT_ID")
client_secret = os.getenv("AZURE_CLIENT_SECRET")

# Use ChainedTokenCredential for robust authentication
if all([tenant_id, client_id, client_secret]):
    credential = ChainedTokenCredential(
        EnvironmentCredential(),
        ClientSecretCredential(tenant_id=tenant_id, client_id=client_id, client_secret=client_secret),
        DefaultAzureCredential()
    )
    logger.info("Using service principal credentials")
else:
    credential = DefaultAzureCredential(exclude_interactive_browser_credential=True)
    logger.info("Using default credentials (managed identity)")

# Get token for Databricks
token = credential.get_token("2ff814a6-3304-4ab8-85cb-cd0e6f879c1d/.default").token
logger.info("✓ Successfully authenticated to Databricks")

In [ ]:
# Create Databricks workspace client
workspace_client = WorkspaceClient(
    host=databricks_host,
    token=token
)

logger.info("✓ Databricks WorkspaceClient initialized")

## 2. Load MLflow Model from Databricks Registry

In [ ]:
# Set MLflow tracking URI to Databricks
mlflow.set_tracking_uri(f"databricks")
mlflow.set_registry_uri(f"databricks://workspace")

logger.info(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
logger.info(f"MLflow registry URI: {mlflow.get_registry_uri()}")

In [ ]:
# Load the model from Databricks MLflow registry
try:
    if databricks_model_version.lower() == "latest":
        model_uri = f"models:/{databricks_model_name}/latest"
    else:
        model_uri = f"models:/{databricks_model_name}/{databricks_model_version}"
    
    logger.info(f"Loading model: {model_uri}")
    model = mlflow.pyfunc.load_model(model_uri)
    
    logger.info(f"✓ Successfully loaded model: {databricks_model_name}")
    logger.info(f"Model type: {type(model)}")
    
except Exception as e:
    logger.error(f"Failed to load model: {e}")
    raise

## 3. Test Model with Sample Data

In [ ]:
# Create sample data for prediction
# Adjust columns based on your model's input schema
sample_data = pd.DataFrame({
    "feature_1": [1.0, 2.0, 3.0],
    "feature_2": [4.0, 5.0, 6.0],
    "feature_3": [7.0, 8.0, 9.0]
})

logger.info(f"Sample data shape: {sample_data.shape}")
logger.info(f"\nSample data:\n{sample_data}")

In [ ]:
# Make predictions using the loaded model
try:
    predictions = model.predict(sample_data)
    
    logger.info(f"✓ Successfully made predictions")
    logger.info(f"Predictions shape: {predictions.shape if hasattr(predictions, 'shape') else len(predictions)}")
    logger.info(f"\nPredictions:\n{predictions}")
    
except Exception as e:
    logger.error(f"Failed to make predictions: {e}")
    raise

## 4. Validate Results

In [ ]:
# Validate predictions
if predictions is not None and len(predictions) == len(sample_data):
    logger.info("✓ Validation successful:")
    logger.info(f"  - Prediction count matches input count (3)")
    logger.info(f"  - Model is callable and returns valid results")
    
    # Summary
    summary = {
        "status": "success",
        "model_name": databricks_model_name,
        "model_version": databricks_model_version,
        "databricks_host": databricks_host,
        "predictions_made": len(predictions),
        "message": f"Successfully loaded and tested model '{databricks_model_name}' from Databricks MLflow registry"
    }
else:
    summary = {
        "status": "validation_failed",
        "message": "Predictions count does not match input count"
    }

logger.info(f"\n=== Integration Test Summary ===")
logger.info(json.dumps(summary, indent=2))